# Phần 1: Đọc file, gộp dữ liệu & tổng hợp bảng

1. **Đọc file & xử lý cột** (đọc CSV, xem thông tin, xoá cột, đổi tên/tạo cột)
2. **Gộp dữ liệu từ nhiều file** (`concat` để nối theo hàng, `merge` để nối theo khoá)
3. **Gộp dữ liệu từ nhiều dòng thành 1 bảng tổng hợp** (`groupby`, `pivot_table`)


## Bước 0 — Chuẩn bị dữ liệu mẫu

Chạy ô bên dưới **trước tiên** để tạo ra các file CSV mẫu trong thư mục `data/`:
- `sales_thang1.csv`, `sales_thang2.csv`, `sales_thang3.csv`: dữ liệu bán hàng theo từng tháng (để luyện gộp file)
- `khach_hang.csv`: danh sách khách hàng
- `don_hang.csv`: danh sách đơn hàng (để luyện `merge`)
- `giao_dich.csv`: dữ liệu giao dịch dạng "dài" — mỗi khách hàng có **nhiều dòng** (để luyện gộp nhiều dòng thành 1 bảng tổng hợp)

In [4]:
import pandas as pd
import numpy as np
import os

os.makedirs('data', exist_ok=True)
np.random.seed(42)

products = ['Áo thun', 'Quần jeans', 'Giày sneaker', 'Mũ lưỡi trai', 'Túi xách']
regions = ['Hà Nội', 'TP.HCM', 'Đà Nẵng', 'Cần Thơ']

def make_sales(month, n=15):
    df = pd.DataFrame({
        'ma_don_hang': [f'{month}-{i:03d}' for i in range(1, n + 1)],
        'san_pham': np.random.choice(products, n),
        'khu_vuc': np.random.choice(regions, n),
        'so_luong': np.random.randint(1, 10, n),
        'don_gia': np.random.choice([150000, 250000, 320000, 450000, 180000], n),
        'ghi_chu': ['' for _ in range(n)]
    })
    return df

make_sales('T1').to_csv('data/sales_thang1.csv', index=False)
make_sales('T2').to_csv('data/sales_thang2.csv', index=False)
make_sales('T3').to_csv('data/sales_thang3.csv', index=False)

customers = pd.DataFrame({
    'ma_kh': [f'KH{i:03d}' for i in range(1, 11)],
    'ten_kh': ['Nguyễn An', 'Trần Bình', 'Lê Chi', 'Phạm Dũng', 'Hoàng Em',
               'Vũ Phong', 'Đặng Giang', 'Bùi Hoa', 'Ngô Khoa', 'Đỗ Linh'],
    'thanh_pho': np.random.choice(regions, 10)
})
customers.to_csv('data/khach_hang.csv', index=False)

orders = pd.DataFrame({
    'ma_don': [f'DH{i:03d}' for i in range(1, 21)],
    'ma_kh': np.random.choice(customers['ma_kh'], 20),
    'san_pham': np.random.choice(products, 20),
    'so_luong': np.random.randint(1, 5, 20),
    'don_gia': np.random.choice([150000, 250000, 320000, 450000], 20)
})
orders.to_csv('data/don_hang.csv', index=False)

n_trans = 40
transactions = pd.DataFrame({
    'ma_kh': np.random.choice(customers['ma_kh'], n_trans),
    'thang': np.random.choice(['T1', 'T2', 'T3'], n_trans),
    'doanh_thu': np.random.randint(100000, 2000000, n_trans)
})
transactions.to_csv('data/giao_dich.csv', index=False)

print("Đã tạo xong dữ liệu mẫu trong thư mục 'data/'")
print(os.listdir('data'))

Đã tạo xong dữ liệu mẫu trong thư mục 'data/'
['giao_dich.csv', 'sales_thang3.csv', 'khach_hang.csv', 'sales_thang1.csv', 'don_hang.csv', 'sales_thang2.csv']


# Đọc file & xử lý cột

### Bài 1.1 — Đọc file CSV
Đọc file `data/sales_thang1.csv` vào một DataFrame tên `df1`.
In ra:
- kích thước (số dòng, số cột) bằng `.shape`
- 5 dòng đầu tiên bằng `.head()`

In [9]:
import pandas as pd
df1 = pd.read_csv('data/sales_thang1.csv')
df1.shape


(15, 6)

In [11]:
df1.head()

,ma_don_hang,san_pham,khu_vuc,so_luong,don_gia,ghi_chu
0,T1-001,Mũ lưỡi trai,TP.HCM,3,250000,NaN
1,T1-002,Túi xách,TP.HCM,7,150000,NaN
2,T1-003,Giày sneaker,TP.HCM,4,250000,NaN
3,T1-004,Túi xách,Cần Thơ,9,180000,NaN
4,T1-005,Túi xách,Cần Thơ,3,250000,NaN


### Bài 1.2 — Xem thông tin tổng quan của dữ liệu
Dùng `.info()` để xem kiểu dữ liệu (dtype) của từng cột và số lượng giá trị không rỗng.
Dùng `.columns` để in ra danh sách tên cột.

In [12]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ma_don_hang  15 non-null     object 
 1   san_pham     15 non-null     object 
 2   khu_vuc      15 non-null     object 
 3   so_luong     15 non-null     int64  
 4   don_gia      15 non-null     int64  
 5   ghi_chu      0 non-null      float64
dtypes: float64(1), int64(2), object(3)
memory usage: 852.0+ bytes


In [14]:
df1.columns

Index(['ma_don_hang', 'san_pham', 'khu_vuc', 'so_luong', 'don_gia', 'ghi_chu'], dtype='object')

### Bài 1.3 — Xoá cột không cần thiết
Cột `ghi_chu` trong `df1` toàn giá trị rỗng, không mang thông tin hữu ích.
Hãy **xoá cột này** bằng `.drop(columns=...)` và gán kết quả lại vào `df1`.

In [15]:
df1 = df1.drop(columns= 'ghi_chu')   # TODO: điền tên cột cần xoá (dạng list)
df1.head()

,ma_don_hang,san_pham,khu_vuc,so_luong,don_gia
0,T1-001,Mũ lưỡi trai,TP.HCM,3,250000
1,T1-002,Túi xách,TP.HCM,7,150000
2,T1-003,Giày sneaker,TP.HCM,4,250000
3,T1-004,Túi xách,Cần Thơ,9,180000
4,T1-005,Túi xách,Cần Thơ,3,250000


### Bài 1.4 — Đổi tên cột & tạo cột mới
1. Đổi tên cột `ma_don_hang` thành `ma_don` bằng `.rename(columns={...})`.
2. Tạo thêm cột mới `thanh_tien = so_luong * don_gia`.

In [16]:
df1 = df1.rename(columns={'ma_don_hang' : 'ma_don'})   # TODO: đổi tên cột ma_don_hang -> ma_don
df1['thanh_tien'] = df1['so_luong'] * df1['don_gia']   # TODO: đặt tên cột mới là 'thanh_tien'
df1.head()

,ma_don,san_pham,khu_vuc,so_luong,don_gia,thanh_tien
0,T1-001,Mũ lưỡi trai,TP.HCM,3,250000,750000
1,T1-002,Túi xách,TP.HCM,7,150000,1050000
2,T1-003,Giày sneaker,TP.HCM,4,250000,1000000
3,T1-004,Túi xách,Cần Thơ,9,180000,1620000
4,T1-005,Túi xách,Cần Thơ,3,250000,750000


# Phần 2️ — Gộp dữ liệu từ nhiều file

### Bài 2.1 — Nối nhiều file cùng cấu trúc (`concat`)
Có 3 file bán hàng theo 3 tháng: `sales_thang1.csv`, `sales_thang2.csv`, `sales_thang3.csv` — cùng cấu trúc cột.

Yêu cầu:
1. Đọc cả 3 file, mỗi file bỏ luôn cột `ghi_chu` khi đọc xong (dùng `.drop`).
2. Dùng `pd.concat([...], ignore_index=True)` để nối 3 bảng lại thành 1 bảng `sales_all`.
3. In ra `sales_all.shape` để kiểm tra tổng số dòng (phải bằng tổng 3 file cộng lại).

In [17]:
d1 = pd.read_csv('data/sales_thang1.csv').drop(columns=['ghi_chu'])
d2 = pd.read_csv('data/sales_thang2.csv').drop(columns=['ghi_chu'])
d3 = pd.read_csv('data/sales_thang3.csv').drop(columns=['ghi_chu'])

sales_all = pd.concat([d1, d2, d3], ignore_index=True)   # TODO: dùng pd.concat để nối 3 bảng
print(sales_all.shape)
sales_all.head()

(45, 5)


,ma_don_hang,san_pham,khu_vuc,so_luong,don_gia
0,T1-001,Mũ lưỡi trai,TP.HCM,3,250000
1,T1-002,Túi xách,TP.HCM,7,150000
2,T1-003,Giày sneaker,TP.HCM,4,250000
3,T1-004,Túi xách,Cần Thơ,9,180000
4,T1-005,Túi xách,Cần Thơ,3,250000


### Bài 2.2 — Nối dữ liệu theo khoá chung (`merge`)
File `don_hang.csv` chứa các đơn hàng, mỗi đơn có `ma_kh` (mã khách hàng) nhưng **không có tên/thành phố** khách hàng.
File `khach_hang.csv` chứa thông tin chi tiết khách hàng.

Yêu cầu: dùng `.merge()` để nối 2 bảng theo cột chung `ma_kh` (kiểu `left join`), tạo bảng `orders_full` có đầy đủ `ten_kh`, `thanh_pho`.

In [18]:
orders = pd.read_csv('data/don_hang.csv')
customers = pd.read_csv('data/khach_hang.csv')

orders_full = orders.merge(customers, on='ma_kh', how= 'left')   # TODO: merge theo 'ma_kh', kiểu 'left'
orders_full.head()

,ma_don,ma_kh,san_pham,so_luong,don_gia,ten_kh,thanh_pho
0,DH001,KH001,Túi xách,1,450000,Nguyễn An,TP.HCM
1,DH002,KH002,Mũ lưỡi trai,1,250000,Trần Bình,Đà Nẵng
2,DH003,KH002,Quần jeans,4,450000,Trần Bình,Đà Nẵng
3,DH004,KH006,Giày sneaker,4,150000,Vũ Phong,Hà Nội
4,DH005,KH007,Áo thun,3,450000,Đặng Giang,Hà Nội


### Bài 2.3 — Kết hợp: tính thành tiền rồi tổng hợp theo khu vực
Từ bảng `sales_all` (đã gộp ở Bài 2.1):
1. Tạo cột `thanh_tien = so_luong * don_gia`.
2. Dùng `.groupby('khu_vuc')['thanh_tien'].sum()` để tính tổng doanh thu theo từng khu vực.
3. Sắp xếp kết quả giảm dần để biết khu vực nào bán chạy nhất.

In [19]:
sales_all['thanh_tien'] = sales_all['so_luong'] * sales_all['don_gia']   # TODO: đặt tên cột 'thanh_tien'
doanh_thu_khu_vuc = sales_all.groupby('khu_vuc')['thanh_tien'].sum().sort_values(ascending=False)
# TODO: dùng groupby theo 'khu_vuc', rồi sum(), sắp xếp giảm dần (ascending=False)
doanh_thu_khu_vuc

,thanh_tien
khu_vuc,
Hà Nội,25570000
TP.HCM,16970000
Cần Thơ,16470000
Đà Nẵng,11620000


# Phần 3️ — Gộp dữ liệu từ nhiều dòng thành 1 bảng tổng hợp

### Bài 3.1 — Gộp nhiều dòng của cùng 1 khách hàng (`groupby` + `agg`)
File `giao_dich.csv` có dạng **"dài"**: mỗi khách hàng (`ma_kh`) xuất hiện ở **nhiều dòng** khác nhau (mỗi lần giao dịch là 1 dòng, có cột `thang` và `doanh_thu`).

Yêu cầu: gộp các dòng theo `ma_kh` để tạo bảng tổng hợp `tong_hop` gồm:
- `tong_doanh_thu`: tổng `doanh_thu` của khách hàng đó
- `so_giao_dich`: số lần giao dịch (đếm số dòng)

Gợi ý: dùng `.groupby('ma_kh').agg(ten_cot_moi=('cot_goc', 'ham'))`.

In [ ]:
gd = pd.read_csv('data/giao_dich.csv')

tong_hop = gd.____('ma_kh').agg(
    tong_doanh_thu=('doanh_thu', ____),   # TODO: hàm tổng
    so_giao_dich=('doanh_thu', ____)      # TODO: hàm đếm
).reset_index()

tong_hop

In [21]:
giao_dich = pd.read_csv('data/giao_dich.csv')
giao_dich.head(10)

,ma_kh,thang,doanh_thu
0,KH003,T2,368246
1,KH003,T2,1854272
2,KH004,T2,218015
3,KH008,T2,1889250
4,KH006,T2,354079
5,KH008,T1,650929
6,KH001,T3,789944
7,KH008,T2,788105
8,KH004,T3,613758
9,KH001,T3,1083609


In [30]:
gd = pd.read_csv('data/giao_dich.csv')

tong_hop = gd.groupby('ma_kh').agg(
    tong_doanh_thu=('doanh_thu', 'sum'),   # TODO: hàm tổng
    so_giao_dich=('doanh_thu', 'count')      # TODO: hàm đếm
).reset_index()

tong_hop

,ma_kh,tong_doanh_thu,so_giao_dich
0,KH001,6801423,6
1,KH002,4406969,3
2,KH003,7734998,7
3,KH004,7167930,7
4,KH005,1541578,1
5,KH006,4175558,4
6,KH007,2381492,2
7,KH008,6045898,7
8,KH009,3643901,3


### Bài 3.2 — Biến dữ liệu "dài" thành "rộng" (`pivot_table`)
Từ bảng `gd` ở trên, mỗi khách hàng có nhiều dòng theo từng tháng (`T1`, `T2`, `T3`).

Yêu cầu: dùng `.pivot_table()` để tạo bảng mới trong đó:
- mỗi khách hàng (`ma_kh`) chỉ còn **1 dòng duy nhất**
- mỗi tháng (`thang`) trở thành **1 cột riêng**
- giá trị trong bảng là **tổng doanh thu** của khách hàng đó trong tháng đó
- nếu khách hàng không có giao dịch trong tháng nào đó thì điền `0` (dùng `fill_value=0`)

In [ ]:
pivot = gd.pivot_table(
    index='ma_kh',       # TODO: cột dùng làm mỗi dòng (khách hàng)
    columns='thang',     # TODO: cột dùng để tách thành các cột mới (tháng)
    values='doanh_thu',      # TODO: cột giá trị cần tổng hợp
    aggfunc='sum',     # TODO: cách tổng hợp ('sum')
    fill_value=0   # TODO: giá trị điền khi thiếu dữ liệu
).reset_index()

pivot

thang,ma_kh,T1,T2,T3
0,KH001,3370209,258823,3172391
1,KH002,2436184,0,1970785
2,KH003,4547029,2222518,965451
3,KH004,3297100,218015,3652815
4,KH005,1541578,0,0
5,KH006,1592785,2582773,0
6,KH007,2381492,0,0
7,KH008,1386423,4659475,0
8,KH009,1152590,1324665,1166646


### Bài 3.3 — Bài tổng hợp cuối cùng
Kết hợp toàn bộ kỹ năng đã học:
1. Từ bảng `pivot` (Bài 3.2), tạo thêm cột `tong = T1 + T2 + T3` (tổng doanh thu cả 3 tháng).
2. `merge` với bảng `customers` theo `ma_kh` để có thêm `ten_kh`, `thanh_pho`.
3. Sắp xếp kết quả theo `tong` giảm dần để tìm ra **khách hàng chi tiêu nhiều nhất**.

In [35]:
pivot['tong'] = pivot['T1'] + pivot['T2'] + pivot['T3']   # TODO: đặt tên cột 'tong'

ket_qua = pivot.merge(customers, on='ma_kh', how= 'left')   # TODO: merge với customers theo 'ma_kh'
ket_qua = ket_qua.sort_values('tong', ascending= False)   # TODO: sắp xếp theo 'tong', giảm dần

ket_qua

,ma_kh,T1,T2,T3,tong,ten_kh,thanh_pho
2,KH003,4547029,2222518,965451,7734998,Lê Chi,TP.HCM
3,KH004,3297100,218015,3652815,7167930,Phạm Dũng,Đà Nẵng
0,KH001,3370209,258823,3172391,6801423,Nguyễn An,TP.HCM
7,KH008,1386423,4659475,0,6045898,Bùi Hoa,Cần Thơ
1,KH002,2436184,0,1970785,4406969,Trần Bình,Đà Nẵng
5,KH006,1592785,2582773,0,4175558,Vũ Phong,Hà Nội
8,KH009,1152590,1324665,1166646,3643901,Ngô Khoa,Hà Nội
6,KH007,2381492,0,0,2381492,Đặng Giang,Hà Nội
4,KH005,1541578,0,0,1541578,Hoàng Em,Hà Nội


In [38]:
cs = pd.read_csv('data/khach_hang.csv')
cs.head()

,ma_kh,ten_kh,thanh_pho
0,KH001,Nguyễn An,TP.HCM
1,KH002,Trần Bình,Đà Nẵng
2,KH003,Lê Chi,TP.HCM
3,KH004,Phạm Dũng,Đà Nẵng
4,KH005,Hoàng Em,Hà Nội


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
Qua notebook này bạn đã luyện tập:
- Đọc file CSV, xem thông tin dữ liệu, xoá/đổi tên/tạo cột
- Gộp nhiều file cùng cấu trúc bằng `concat`
- Nối dữ liệu theo khoá chung bằng `merge`
- Gộp nhiều dòng thành 1 bảng tổng hợp bằng `groupby`/`agg` và `pivot_table`
